# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [1]:
import numpy as np

query = "latency vs throughput"
docs = [f"doc{i}: ..." for i in range(5)]
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])  
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3])  

# 1. Tomar top-4 por bi-encoder (los índices con mayores scores)
top4_indices = np.argsort(bi_scores)[-4:][::-1]

# 2. Reordenar esos 4 usando los scores del cross-encoder
final_order = sorted(top4_indices, key=lambda i: cross_scores[i], reverse=True)

print("Orden original:", docs)
print("Orden tras Reranking:")
for i in final_order:
    print(f"Doc {i} (Cross-Score: {cross_scores[i]})")

Orden original: ['doc0: ...', 'doc1: ...', 'doc2: ...', 'doc3: ...', 'doc4: ...']
Orden tras Reranking:
Doc 1 (Cross-Score: 0.9)
Doc 3 (Cross-Score: 0.85)
Doc 4 (Cross-Score: 0.3)
Doc 0 (Cross-Score: 0.1)


## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [2]:
# Pseudocódigo de selección MMR
"""
def MMR_select(query_emb, doc_embs, top_k, lambda_param=0.5):
    selected_docs = []
    unselected_docs = list(range(len(doc_embs)))
    
    while len(selected_docs) < top_k:
        best_score = -inf
        best_doc = None
        for doc in unselected_docs:
            sim_to_query = similitud(query_emb, doc_embs[doc])
            max_sim_to_selected = max([similitud(doc_embs[doc], doc_embs[s]) for s in selected_docs]) if selected_docs else 0
            
            # Ecuación de balance
            mmr_score = (lambda_param * sim_to_query) - ((1 - lambda_param) * max_sim_to_selected)
            
            if mmr_score > best_score:
                best_score = mmr_score
                best_doc = doc
                
        selected_docs.append(best_doc)
        unselected_docs.remove(best_doc)
        
    return selected_docs
"""

'\ndef MMR_select(query_emb, doc_embs, top_k, lambda_param=0.5):\n    selected_docs = []\n    unselected_docs = list(range(len(doc_embs)))\n\n    while len(selected_docs) < top_k:\n        best_score = -inf\n        best_doc = None\n        for doc in unselected_docs:\n            sim_to_query = similitud(query_emb, doc_embs[doc])\n            max_sim_to_selected = max([similitud(doc_embs[doc], doc_embs[s]) for s in selected_docs]) if selected_docs else 0\n\n            # Ecuación de balance\n            mmr_score = (lambda_param * sim_to_query) - ((1 - lambda_param) * max_sim_to_selected)\n\n            if mmr_score > best_score:\n                best_score = mmr_score\n                best_doc = doc\n\n        selected_docs.append(best_doc)\n        unselected_docs.remove(best_doc)\n\n    return selected_docs\n'

## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa | Coste relativo (tú eliges escala) |
|--------|-------------------------------------|
| Etapa | Coste relativo (Escala: Bajo / Medio / Alto) |
|--------|-------------------------------------|
| Embedding único de query | Bajo (Rápido, se procesa un vector pequeño) |
| K llamadas Cross-Encoder | Medio-Alto (Lento, se hace inferencia conjunta Query+Doc $K$ veces) |
| Generación LLM 200 tokens| Muy Alto (Inferencia auto-regresiva pesada y costosa por token) |